# Neural–DTB score transport: a self-contained Cournot example

This Colab notebook implements the discrete Neural–DTB algorithm in the supplied equations. It tracks particles $x_i^k$, log densities $\ell_i^k$, and scores $q_i^k=\nabla_x\log\rho_k(x_i^k)$. Every function is commented, and the last sections visualize and validate the score computation.

The notebook uses only PyTorch, NumPy, and Matplotlib, which are already installed in Google Colab. A GPU is used when available.

## 1. Equations implemented

At step $k$, the score-corrected target is

$$v_i^k=b(x_i^k)-\tfrac12Dq_i^k.$$

For $J_i^k=\partial_\theta f_\theta(x_i^k)$, the code assembles $G_k=\sum_iJ_i^\top J_i$ and $P_k=\sum_iJ_i^\top v_i$, then solves $G_k\alpha_k=P_k$ through a truncated SVD of the stacked least-squares system. The projected action is $u_k(x)=\partial_\theta f_\theta(x)\alpha_k$.

The explicit Euler state update is

$$x^{k+1}=x^k+hu_k(x^k),\qquad \ell^{k+1}=\ell^k-h\,\nabla\!\cdot u_k(x^k),$$
$$q^{k+1}=q^k-h\left([\nabla u_k(x^k)]^\top q^k+\nabla(\nabla\!\cdot u_k)(x^k)\right).$$

The integral expression $P_k=\int[J^\top b+\tfrac12\nabla_x\cdot(D^\top J)]\,d\lambda$ is the integration-by-parts version of the score-based expression. Section 6 compares the two Monte Carlo estimators.

In [ ]:
# Core imports. Colab already includes these packages.
import math
import time
from dataclasses import dataclass
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import functional_call, jacrev, vmap

SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)

# CUDA is substantially faster in Colab. CPU is the most portable fallback.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
if DEVICE.type == "cuda":
    torch.set_float32_matmul_precision("high")

print(f"PyTorch {torch.__version__} | device={DEVICE} | dtype={DTYPE}")

## 2. Multi-player Cournot game

Let $r_i=\sum_{j\ne i}x_j$. The repository's equilibrium-consistent non-potential Cournot example uses

$$\Pi_i(x)=-b x_i^2+2b\mu x_i r_i(1-r_i),$$
$$b_i(x)=\partial_{x_i}\Pi_i=-2b x_i+2b\mu r_i-2b\mu r_i^2.$$

For two players with $b=1$ and $\mu=2$, $(0,0)$ and $(1/2,1/2)$ are equilibria, and the latter is stable.

Equation (4.39) in the screenshot can also be read literally as $a-b\sum_jx_j-2b\mu x_i r_i^2$. Its own-action gradient is $-b-2b\mu r_i^2$, which gives qualitatively different boundary behavior. Both versions are coded below; change `USE_LITERAL_PRINTED_PAYOFF` if that literal convention is intended.

In [ ]:
def cournot_payoff(x: torch.Tensor, price_slope: float = 1.0, mu: float = 2.0) -> torch.Tensor:
    """Return one payoff per player for the equilibrium-consistent game."""
    opponents = x.sum(dim=-1, keepdim=True) - x
    return (-price_slope * x.square()
            + 2.0 * price_slope * mu * x * opponents * (1.0 - opponents))


def cournot_drift(x: torch.Tensor, price_slope: float = 1.0, mu: float = 2.0) -> torch.Tensor:
    """Compute b_i=partial Pi_i/partial x_i for all particles and players."""
    opponents = x.sum(dim=-1, keepdim=True) - x
    return (-2.0 * price_slope * x
            + 2.0 * price_slope * mu * opponents
            - 2.0 * price_slope * mu * opponents.square())


def literal_printed_payoff(x: torch.Tensor, a: float = 1.0,
                           price_slope: float = 1.0, mu: float = 2.0) -> torch.Tensor:
    """Implement equation (4.39) exactly as it appears in the screenshot."""
    total = x.sum(dim=-1, keepdim=True)
    opponents = total - x
    return a - price_slope * total - 2.0 * price_slope * mu * x * opponents.square()


def literal_printed_drift(x: torch.Tensor, price_slope: float = 1.0,
                          mu: float = 2.0) -> torch.Tensor:
    """Own-action gradient of the literal printed payoff."""
    opponents = x.sum(dim=-1, keepdim=True) - x
    return -price_slope - 2.0 * price_slope * mu * opponents.square()


def own_action_gradient(payoff_fn: Callable[[torch.Tensor], torch.Tensor],
                        x: torch.Tensor) -> torch.Tensor:
    """Use autodiff to extract the diagonal (own-action) payoff derivatives."""
    def one_profile(profile: torch.Tensor) -> torch.Tensor:
        payoff_jacobian = jacrev(payoff_fn)(profile)
        return payoff_jacobian.diagonal()
    return vmap(one_profile)(x)


USE_LITERAL_PRINTED_PAYOFF = False  # @param {type:"boolean"}
DRIFT = literal_printed_drift if USE_LITERAL_PRINTED_PAYOFF else cournot_drift

# A small unit check prevents a silent algebra error in the default pseudo-gradient.
probe = torch.tensor([[0.2, 0.4], [0.5, 0.5]], device=DEVICE, dtype=DTYPE)
autodiff_drift = own_action_gradient(cournot_payoff, probe)
assert torch.allclose(autodiff_drift, cournot_drift(probe), atol=1e-6)
print("Cournot pseudo-gradient check passed. Values:\n", cournot_drift(probe))

## 3. Particle state and initial score

A Gaussian initial law makes the score nonzero and known exactly. For $\rho_0=\mathcal N(m,s^2I)$,

$$\log\rho_0(x)=-\frac{\|x-m\|^2}{2s^2}-d\log s-\frac d2\log(2\pi),\qquad q^0(x)=-\frac{x-m}{s^2}.$$

In [ ]:
@dataclass
class ParticleState:
    """Store the particle, log-density, score, and immutable reference labels."""
    particles: torch.Tensor  # shape (N,d)
    log_density: torch.Tensor  # shape (N,)
    score: torch.Tensor  # shape (N,d)
    labels: torch.Tensor  # fixed z_i, shape (N,d)

    def validate(self) -> None:
        """Fail early if one state component has an inconsistent shape."""
        n, dim = self.particles.shape
        assert self.log_density.shape == (n,)
        assert self.score.shape == (n, dim)
        assert self.labels.shape == (n, dim)
        assert all(torch.isfinite(t).all() for t in
                   (self.particles, self.log_density, self.score))


def gaussian_log_density_and_score(x: torch.Tensor, mean: torch.Tensor,
                                   std: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Evaluate log rho_0 and its analytic score at a batch of points."""
    centered = x - mean
    dim = x.shape[-1]
    log_density = (-0.5 * centered.square().sum(dim=-1) / std**2
                   - dim * math.log(std) - 0.5 * dim * math.log(2.0 * math.pi))
    score = -centered / std**2
    return log_density, score


def sample_gaussian_state(n: int, mean: torch.Tensor, std: float) -> ParticleState:
    """Sample z~lambda, use X_0(z)=z, and initialize ell^0 and q^0."""
    labels = mean + std * torch.randn(n, mean.numel(), device=mean.device, dtype=mean.dtype)
    particles = labels.clone()
    log_density, score = gaussian_log_density_and_score(particles, mean, std)
    state = ParticleState(particles, log_density, score, labels)
    state.validate()
    return state

## 4. Neural map and parameter Jacobian

For a compact, fast example, $f_\theta$ is an identity residual map with fixed smooth random features and a trainable readout: $f_\theta(x)=x+W_\theta\tanh(Ax+c)$. It is a one-hidden-layer neural map and is linear in its trainable parameters. Starting with $W_\theta=0$ gives $f_\theta(x)=x=X_0(x)$.

The parameter Jacobian is still computed with `torch.func.jacrev`, so this cell is a useful template for replacing the model with a fully trainable MLP. Because this example is linear in its only trainable parameters, its tangent basis does not change with $\theta$ and no reset schedule is needed.

In [ ]:
class RandomFeatureNeuralMap(nn.Module):
    """Identity residual map with a fixed tanh layer and trainable readout."""
    def __init__(self, dim: int, width: int):
        super().__init__()
        self.dim = dim
        self.width = width
        self.register_buffer("feature_weight", 1.5 * torch.randn(width, dim))
        self.register_buffer("feature_bias", 2.0 * math.pi * torch.rand(width) - math.pi)
        self.readout = nn.Parameter(torch.zeros(dim, width))

    def features(self, x: torch.Tensor) -> torch.Tensor:
        """Evaluate the fixed nonlinear feature layer."""
        return torch.tanh(F.linear(x, self.feature_weight, self.feature_bias))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Evaluate f_theta(x)."""
        return x + F.linear(self.features(x), self.readout)

    def tangent_velocity(self, x: torch.Tensor, alpha: torch.Tensor) -> torch.Tensor:
        """Evaluate u(x)=J(x)alpha for this readout-linear architecture."""
        alpha_matrix = alpha.reshape_as(self.readout)
        return F.linear(self.features(x), alpha_matrix)


def flat_parameters(model: RandomFeatureNeuralMap) -> torch.Tensor:
    """Return the trainable readout as one vector theta."""
    return model.readout.detach().reshape(-1)


def map_at_flat_parameters(model: RandomFeatureNeuralMap, theta: torch.Tensor,
                           x_single: torch.Tensor) -> torch.Tensor:
    """Functional model call used by parameter autodifferentiation."""
    parameter_dict = {"readout": theta.reshape_as(model.readout)}
    return functional_call(model, parameter_dict, (x_single.unsqueeze(0),)).squeeze(0)


def parameter_jacobian(model: RandomFeatureNeuralMap, theta: torch.Tensor,
                       particles: torch.Tensor) -> torch.Tensor:
    """Return J_i=partial_theta f_theta(x_i) with shape (N,d,M)."""
    jacobian_one = jacrev(lambda th, xx: map_at_flat_parameters(model, th, xx), argnums=0)
    return vmap(jacobian_one, in_dims=(None, 0))(theta, particles)

## 5. Projection solve and spatial derivatives

The code explicitly forms $G$ and $P$ for inspection, but solves the equivalent stacked least-squares problem by truncated SVD. This avoids squaring the condition number as a direct normal-equation solve would. All derivatives needed by the score update are obtained by automatic differentiation.

In [ ]:
@dataclass
class ProjectionInfo:
    """Numerical diagnostics from one tangent-space projection."""
    rank: int
    relative_residual: float
    normal_equation_residual: float
    sigma_max: float
    sigma_min_retained: float


def solve_projection(jacobians: torch.Tensor, target: torch.Tensor,
                     rtol: float = 1e-5):
    """Assemble G and P, then compute the minimum-norm truncated-SVD solution."""
    n, dim, parameter_count = jacobians.shape
    stacked_j = jacobians.reshape(n * dim, parameter_count)
    stacked_v = target.reshape(n * dim)

    # These are exactly the G_k and P_k sums in the supplied algorithm.
    G = torch.einsum("ndm,ndp->mp", jacobians, jacobians)
    P = torch.einsum("ndm,nd->m", jacobians, target)

    U, singular_values, Vh = torch.linalg.svd(stacked_j, full_matrices=False)
    if singular_values.numel() == 0 or singular_values[0] == 0:
        alpha = torch.zeros(parameter_count, device=target.device, dtype=target.dtype)
        keep = torch.zeros_like(singular_values, dtype=torch.bool)
    else:
        keep = singular_values > rtol * singular_values[0]
        coefficients = (U[:, keep].T @ stacked_v) / singular_values[keep]
        alpha = Vh[keep].T @ coefficients

    residual = torch.linalg.norm(stacked_j @ alpha - stacked_v)
    relative_residual = residual / (torch.linalg.norm(stacked_v) + 1e-12)
    normal_residual = torch.linalg.norm(G @ alpha - P) / (torch.linalg.norm(P) + 1e-12)
    info = ProjectionInfo(
        rank=int(keep.sum()),
        relative_residual=float(relative_residual),
        normal_equation_residual=float(normal_residual),
        sigma_max=float(singular_values[0]) if singular_values.numel() else 0.0,
        sigma_min_retained=float(singular_values[keep][-1]) if keep.any() else 0.0,
    )
    return alpha.detach(), info, G.detach(), P.detach()


def velocity_spatial_terms(model: RandomFeatureNeuralMap, alpha: torch.Tensor,
                           particles: torch.Tensor):
    """Evaluate u, grad u, div u, and grad(div u) at every particle."""
    def u_one(x_single: torch.Tensor) -> torch.Tensor:
        return model.tangent_velocity(x_single, alpha)

    grad_u_one = jacrev(u_one)

    def divergence_one(x_single: torch.Tensor) -> torch.Tensor:
        return torch.trace(grad_u_one(x_single))

    grad_divergence_one = jacrev(divergence_one)
    velocity = vmap(u_one)(particles)
    grad_u = vmap(grad_u_one)(particles)
    divergence = vmap(divergence_one)(particles)
    grad_divergence = vmap(grad_divergence_one)(particles)
    return velocity, grad_u, divergence, grad_divergence

## 6. Score-based and integration-by-parts forms of $P_k$

When $q=\nabla\log\rho$, integration by parts gives

$$\mathbb E_\rho[J^\top(b-\tfrac12Dq)] = \mathbb E_\rho[J^\top b+\tfrac12\nabla_x\cdot(D^\top J)],$$

assuming the boundary term vanishes. The two finite-sample Monte Carlo estimates are not identical, but their difference should shrink with more samples.

In [ ]:
def integration_by_parts_P(model: RandomFeatureNeuralMap, theta: torch.Tensor,
                           particles: torch.Tensor, drift_fn, D: torch.Tensor) -> torch.Tensor:
    """Estimate sum_i [J_i^T b_i + 1/2 div_x(D^T J_i)]."""
    jacobian_one = jacrev(lambda th, xx: map_at_flat_parameters(model, th, xx), argnums=0)

    def divergence_DtJ_one(x_single: torch.Tensor) -> torch.Tensor:
        # D^T J is a d-by-M matrix; take a spatial divergence of each column.
        def DtJ(y: torch.Tensor) -> torch.Tensor:
            return D.T @ jacobian_one(theta, y)
        spatial_derivative = jacrev(DtJ)(x_single)  # shape (d,M,d)
        return torch.einsum("rmr->m", spatial_derivative)

    J = parameter_jacobian(model, theta, particles)
    first_term = torch.einsum("ndm,nd->nm", J, drift_fn(particles))
    divergence_term = vmap(divergence_DtJ_one)(particles)
    return (first_term + 0.5 * divergence_term).sum(dim=0)


# Editable Colab controls. FAST_MODE completes quickly on CPU.
FAST_MODE = True  # @param {type:"boolean"}
DIM = 2
N_PARTICLES = 320 if FAST_MODE else 1200
N_IDENTITY_CHECK = 2048 if FAST_MODE else 12000
WIDTH = 20 if FAST_MODE else 48
K_STEPS = 30 if FAST_MODE else 100
STEP_SIZE = 0.01
SVD_RTOL = 1e-5
NOISE_STD = 0.10
D = (NOISE_STD**2) * torch.eye(DIM, device=DEVICE, dtype=DTYPE)
INITIAL_MEAN = torch.full((DIM,), 0.30, device=DEVICE, dtype=DTYPE)
INITIAL_STD = 0.12

model = RandomFeatureNeuralMap(DIM, WIDTH).to(device=DEVICE, dtype=DTYPE)
theta = flat_parameters(model)

# Use an independent, larger sample for the P identity check.
identity_state = sample_gaussian_state(N_IDENTITY_CHECK, INITIAL_MEAN, INITIAL_STD)
J_check = parameter_jacobian(model, theta, identity_state.particles)
target_check = DRIFT(identity_state.particles) - 0.5 * (identity_state.score @ D.T)
P_score = torch.einsum("ndm,nd->m", J_check, target_check)
P_ibp = integration_by_parts_P(model, theta, identity_state.particles, DRIFT, D)
relative_P_difference = torch.linalg.norm(P_score - P_ibp) / (torch.linalg.norm(P_score) + 1e-12)
print(f"P identity Monte Carlo relative difference: {float(relative_P_difference):.3e}")
print("(This is sampling error, not a sample-by-sample identity.)")

## 7. One Neural–DTB step and the complete run

Every right-hand side below uses the old state $(x^k,\ell^k,q^k)$, exactly as in an explicit Euler method. The labels $z_i$ never change, so the stored particles are evaluations of the pushforward map: $x_i^k=X_k(z_i)$.

In [ ]:
def neural_dtb_step(state: ParticleState, model: RandomFeatureNeuralMap,
                    theta: torch.Tensor, drift_fn, D: torch.Tensor, h: float,
                    svd_rtol: float):
    """Perform one complete score-aware Neural–DTB Euler step."""
    state.validate()
    x_k, ell_k, q_k = state.particles, state.log_density, state.score

    # Algorithm line 6: drift plus the diffusion/score correction.
    target_velocity = drift_fn(x_k) - 0.5 * (q_k @ D.T)

    # Lines 7-10: evaluate J, assemble G/P, solve, and form u=J alpha.
    J = parameter_jacobian(model, theta, x_k)
    alpha, projection_info, G, P = solve_projection(J, target_velocity, rtol=svd_rtol)
    u_k, grad_u, divergence, grad_divergence = velocity_spatial_terms(model, alpha, x_k)

    # Line 13: [grad u]^T q. grad_u[n,a,b] = partial_b u_a.
    transported_score = torch.einsum("nab,na->nb", grad_u, q_k)
    next_state = ParticleState(
        particles=(x_k + h * u_k).detach(),
        log_density=(ell_k - h * divergence).detach(),
        score=(q_k - h * (transported_score + grad_divergence)).detach(),
        labels=state.labels,
    )
    next_state.validate()
    extras = {
        "target_norm": float(torch.linalg.norm(target_velocity) / math.sqrt(target_velocity.numel())),
        "velocity_norm": float(torch.linalg.norm(u_k) / math.sqrt(u_k.numel())),
        "mean_score_norm": float(torch.linalg.vector_norm(next_state.score, dim=1).mean()),
        "mean_divergence": float(divergence.mean()),
        "G_shape": tuple(G.shape),
        "P_shape": tuple(P.shape),
    }
    return next_state, alpha, projection_info, extras


def run_neural_dtb(initial_state: ParticleState, model: RandomFeatureNeuralMap,
                   drift_fn, D: torch.Tensor, steps: int, h: float, svd_rtol: float):
    """Run K steps and retain alphas, diagnostics, and three particle snapshots."""
    theta = flat_parameters(model)
    state = initial_state
    alpha_history, diagnostics = [], []
    snapshot_steps = {0, steps // 2, steps}
    snapshots = {0: state.particles.detach().cpu().clone()}

    for k in range(steps):
        state, alpha, info, extras = neural_dtb_step(
            state, model, theta, drift_fn, D, h, svd_rtol
        )
        alpha_history.append(alpha)
        diagnostics.append({"step": k + 1, **vars(info), **extras})
        if k + 1 in snapshot_steps:
            snapshots[k + 1] = state.particles.detach().cpu().clone()

    return state, alpha_history, diagnostics, snapshots


initial_state = sample_gaussian_state(N_PARTICLES, INITIAL_MEAN, INITIAL_STD)
start = time.perf_counter()
final_state, alpha_history, diagnostics, snapshots = run_neural_dtb(
    initial_state, model, DRIFT, D, K_STEPS, STEP_SIZE, SVD_RTOL
)
elapsed = time.perf_counter() - start

print(f"Completed {K_STEPS} steps with N={N_PARTICLES}, M={theta.numel()} in {elapsed:.2f} s")
print(f"G shape={diagnostics[0]['G_shape']} | P shape={diagnostics[0]['P_shape']}")
print(f"final projection residual={diagnostics[-1]['relative_residual']:.3e}")
print(f"final normal-equation residual={diagnostics[-1]['normal_equation_residual']:.3e}")

## 8. Pushforward and score validation

First, the code replays $X_{k+1}(z)=X_k(z)+hu_k(X_k(z))$ and checks that it reproduces the stored particles. Second, for a few labels it independently differentiates the composed discrete pushforward and applies change of variables:

$$q_K(X_K(z))=[\nabla_zX_K(z)]^{-T}\left(q_0(z)-\nabla_z\log|\det\nabla_zX_K(z)|\right).$$

The transported score uses explicit Euler for the continuous score ODE, whereas this formula differentiates the discrete map exactly, so a small time-discretization difference is expected.

In [ ]:
def pushforward_one(model: RandomFeatureNeuralMap, z: torch.Tensor,
                    alphas: list[torch.Tensor], h: float) -> torch.Tensor:
    """Evaluate the composed discrete map X_K at one reference point."""
    x = z
    for alpha in alphas:
        x = x + h * model.tangent_velocity(x, alpha)
    return x


def score_from_discrete_change_of_variables(model: RandomFeatureNeuralMap,
                                            z: torch.Tensor, alphas, h: float,
                                            mean: torch.Tensor, std: float) -> torch.Tensor:
    """Independently recover the final score by differentiating X_K(z)."""
    map_one = lambda zz: pushforward_one(model, zz, alphas, h)
    jacobian_X = jacrev(map_one)(z)

    def log_abs_det_jacobian(zz: torch.Tensor) -> torch.Tensor:
        _, log_abs_det = torch.linalg.slogdet(jacrev(map_one)(zz))
        return log_abs_det

    grad_log_det = jacrev(log_abs_det_jacobian)(z)
    _, initial_score = gaussian_log_density_and_score(z.unsqueeze(0), mean, std)
    rhs = initial_score.squeeze(0) - grad_log_det
    return torch.linalg.solve(jacobian_X.T, rhs)


N_PROBES = 6
probe_labels = initial_state.labels[:N_PROBES]
replayed = vmap(lambda z: pushforward_one(model, z, alpha_history, STEP_SIZE))(probe_labels)
map_error = torch.max(torch.abs(replayed - final_state.particles[:N_PROBES]))
assert float(map_error) < 5e-5

change_of_variables_score = vmap(
    lambda z: score_from_discrete_change_of_variables(
        model, z, alpha_history, STEP_SIZE, INITIAL_MEAN, INITIAL_STD
    )
)(probe_labels)
score_rmse = torch.sqrt(torch.mean(
    (change_of_variables_score - final_state.score[:N_PROBES])**2
))
score_scale = torch.sqrt(torch.mean(change_of_variables_score**2))
relative_score_rmse = score_rmse / (score_scale + 1e-12)
print(f"pushforward replay max error: {float(map_error):.3e}")
print(f"transported-vs-change-of-variables score RMSE: {float(score_rmse):.3e}")
print(f"relative score RMSE: {float(relative_score_rmse):.3e}")

## 9. Results

The first row shows the evolving particle law and the two equilibria of the default two-player drift. The lower panels report tangent-projection accuracy, retained rank, and score magnitude. Arrows in the last particle panel are the transported score vectors.

In [ ]:
# Particle snapshots with consistent axis limits.
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)
all_points = torch.cat(list(snapshots.values()), dim=0).numpy()
padding = 0.05
low = float(all_points.min()) - padding
high = float(all_points.max()) + padding

for ax, step in zip(axes, sorted(snapshots)):
    points = snapshots[step].numpy()
    ax.scatter(points[:, 0], points[:, 1], s=11, alpha=0.45, edgecolors="none")
    if not USE_LITERAL_PRINTED_PAYOFF:
        ax.scatter([0.0], [0.0], marker="x", s=90, c="goldenrod", label="saddle")
        ax.scatter([0.5], [0.5], marker="*", s=120, c="crimson", label="stable")
    ax.set(title=f"t={step * STEP_SIZE:.2f}", xlabel="player 1 quantity",
           ylabel="player 2 quantity", xlim=(low, high), ylim=(low, high))
    ax.grid(alpha=0.2)

# Overlay a readable subset of score vectors on the final cloud.
stride = max(1, N_PARTICLES // 45)
points = final_state.particles[::stride].detach().cpu().numpy()
scores = final_state.score[::stride].detach().cpu().numpy()
axes[-1].quiver(points[:, 0], points[:, 1], scores[:, 0], scores[:, 1],
                color="black", alpha=0.55, angles="xy", scale_units="xy", scale=110)
if not USE_LITERAL_PRINTED_PAYOFF:
    axes[-1].legend(loc="best")
plt.show()

# Numerical diagnostics over time.
steps = np.array([row["step"] for row in diagnostics])
residuals = np.array([row["relative_residual"] for row in diagnostics])
normal_residuals = np.array([row["normal_equation_residual"] for row in diagnostics])
ranks = np.array([row["rank"] for row in diagnostics])
score_norms = np.array([row["mean_score_norm"] for row in diagnostics])

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), constrained_layout=True)
axes[0].semilogy(steps, np.maximum(residuals, 1e-12), label="projection")
axes[0].semilogy(steps, np.maximum(normal_residuals, 1e-12), label="normal equation")
axes[0].set(title="Projection diagnostics", xlabel="step", ylabel="relative residual")
axes[0].legend()
axes[1].plot(steps, ranks)
axes[1].set(title="Retained tangent rank", xlabel="step", ylabel="rank")
axes[2].plot(steps, score_norms)
axes[2].set(title="Transported score", xlabel="step", ylabel="mean ||q||")
for ax in axes:
    ax.grid(alpha=0.25)
plt.show()

## Adapting the notebook

- Change `DIM` and use `cournot_drift` directly for more players.
- Replace `DRIFT` with any function mapping `(N,d)` states to `(N,d)` pseudo-gradients.
- Replace the Gaussian initializer only if the new density's initial score is known.
- Increase `WIDTH`, `N_PARTICLES`, and `K_STEPS` after the fast run succeeds.
- Decrease `STEP_SIZE` if the score validation error is too large; the state equations use explicit Euler.
- A fully trainable MLP can replace the random-feature map, but then a parameter reset/refit schedule should be added so the tangent linearization remains local.